# Hyperparameter Tuning Sweeps

This notebook demonstrates how to perform hyperparameter sweeps using `StudySweepConfigurator` and Optuna.

## Imports

In [ ]:
import landseg.adapters.api as api

## Essential / Shared Configuration

First, define the common configurations that are shared across all sweeps (e.g., experiment root directory, input dataset name, data patch size, and domain source conditioning).
We also configure default training/runtime settings (like max epochs per trial).

In [ ]:
# Initialize configurator with experiment and dataset paths
configurator = api.StudySweepConfigurator(
    experiment_root='../experiment/',
    dataset_name='demo_data'
)

# Essential: Set patch size (shared across trials/presets)
configurator.set_data_loading(
    batch_size=16, # default starting batch size
    patch_size=128 # patch size must divide tile size
)

# Essential: Set domain source for conditioning
configurator.set_domain_source(
    category_domain='ecodistrict',
    continuous_domain='geology'
)

# Essential: Set common training/runtime configs (e.g., max epochs per trial)
configurator.set_runtime(
    max_epochs=3,
    patience_epoch=None,
    logit_adjust_alpha=0.0
)

## Section 1: Base Objectives Sweep (Smoke Test)

The `'base'` preset is intended as a quick test of the sweep pipeline. It tunes:
- Learning rate (`learning_rate` range)
- Batch size (`batch_size` range)

You can run this section to verify that the sweep is functioning correctly.

In [ ]:
# Configure Base preset sweep settings
configurator.set_sweep(
    study_name='study_base_test',
    preset_name='base',
    n_trials=5,
    storage='sqlite:///optuna.db',
    direction='maximize',
    seed=42
)

# Customize the sweep ranges for this preset (optional)
configurator.set_base_preset_ranges(
    learning_rate=(1e-5, 1e-2),
    batch_size=(16, 32, 16)
)

# Run the sweep pipeline
api.run(configurator.running_root_config)

## Section 2: Entry-Level Sweep

The `'quick'` preset is an entry-level sweep that tunes optimization and throughput parameters:
- Optimizer learning rate (`learning_rate` range)
- Optimizer weight decay (`weight_decay` range)
- DataLoader batch size (`batch_size` range)
- Automatic mixed precision enablement (`use_amp` list options)

In [ ]:
# Configure Entry-Level preset sweep settings
configurator.set_sweep(
    study_name='study_entry_level',
    preset_name='quick',
    n_trials=15,
    storage='sqlite:///optuna.db',
    direction='maximize',
    seed=42
)

# Customize the sweep ranges for this preset (optional)
configurator.set_quick_preset_ranges(
    learning_rate=(1e-5, 1e-2),
    weight_decay=(1e-6, 1e-3),
    batch_size=(16, 64, 16),
    use_amp=[True, False]
)

# Run the sweep pipeline
api.run(configurator.running_root_config)

## Section 3: Moderate-Level Sweep

The `'capacity'` preset is a moderate-level sweep focusing on model architecture capacity and bottlenecks. It tunes:
- Backbone body (`model_body` list options)
- Backbone channels (`base_channel` range)
- Bottleneck type (`bottleneck` list options)
- Number of convolution & transformer blocks
- Transformer params (attention heads, mlp ratio, dropout, etc.)

In [ ]:
# Configure Moderate-Level preset sweep settings
configurator.set_sweep(
    study_name='study_moderate_level',
    preset_name='capacity',
    n_trials=30,
    storage='sqlite:///optuna.db',
    direction='maximize',
    seed=42
)

# Customize the sweep ranges for this preset (optional)
configurator.set_capacity_preset_ranges(
    model_body=['unet', 'unetpp'],
    base_channel=(16, 32, 16),
    bottleneck=['conv', 'transformer'],
    num_conv_blocks=(1, 3, 1),
    num_transformer_blocks=(1, 3, 1),
    num_heads=[2, 4],
    mlp_ratio=(1.0, 3.0),
    dropout=(0.0, 0.3),
    attn_dropout=(0.0, 0.3)
)

# Run the sweep pipeline
api.run(configurator.running_root_config)